# 🛠️ Step 2: Data Preprocessing & Text Normalization
### Project: Consumer Financial Complaints Classification
**Notebook Goal:** Transform raw, noisy text into clean, structured data ready for Machine Learning & Transformer models.

---
### Preprocessing Pipeline:
1. **EDA Filter Application:** Filter to Top 10 Product categories & drop missing/duplicate narratives.
2. **CFPB Anonymization Handling:** Clean redact tokens (`XXXX`, `XX/XX/XXXX`) and noise.
3. **Text Normalization:** Lowercasing, punctuation removal, regex cleaning, and whitespace trimming.
4. **Target Label Encoding:** Map categorical target classes into numeric integers ($0$ to $9$).
5. **Stratified Train-Validation-Test Split:** Preserve class ratios across all data partitions.

In [2]:
import re
import string
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Settings
pd.set_option('display.max_columns', None)
print("Preprocessing libraries imported successfully!")

Preprocessing libraries imported successfully!


In [3]:
url = "https://huggingface.co/datasets/milesbutler/consumer_complaints/resolve/main/data/train-00000-of-00001.parquet?download=true"
df = pd.read_parquet(url)

text_col = 'Consumer Complaint'
target_col = 'Product'


df_clean = df[df[text_col].notnull()].copy()

top_10_categories = df_clean[target_col].value_counts().head(10).index.tolist()
df_clean = df_clean[df_clean[target_col].isin(top_10_categories)].copy()

# Drop Exact duplicates 
initial_len = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=[text_col]).copy()
dropped_dupes = initial_len - len(df_clean)

print(f"Total Filtered Complaints : {len(df_clean):,}")
print(f"Dropped Duplicate Records : {dropped_dupes:,}")
print(f"Target Classes Retained   : {df_clean[target_col].nunique()}")

Total Filtered Complaints : 192,636
Dropped Duplicate Records : 5,861
Target Classes Retained   : 10


## 2. Text Cleaning & Anonymization Removal
Building a custom regex cleaning function to:
- Convert text to lowercase.
- Remove CFPB redaction markers (`XXXX`, `XX/XX/XXXX`).
- Remove URLs, web links, and email addresses.
- Remove special characters and digits.
- Normalize extra whitespaces.

In [4]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Lowercase 
    text = text.lower()
    
    # 2. remove URLs and web links 
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # 3. Remove anonymized tokens of CFPB (like 'xxxx', 'xx/xx/xxxx')
    text = re.sub(r'\b[x]{2,}\b|\bxx[/\-]\S+', '', text)
    
    # 4. Remove Numbers and special symbols 
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    # 5. Convert Multiple spaces into single space and strip the spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Demo sample
sample = "On XX/XX/2018, Bank of America charged $50.00 fee on my account XXXX! Visit https://bank.com"
print("Original Text :", sample)
print("Cleaned Text  :", clean_text(sample))

Original Text : On XX/XX/2018, Bank of America charged $50.00 fee on my account XXXX! Visit https://bank.com
Cleaned Text  : on bank of america charged fee on my account visit


In [5]:
# Apply on the dataset
df_clean['cleaned_text'] = df_clean[text_col].apply(clean_text)

# Check karein ki clean hone ke baad koi text empty toh nahi ho gaya
empty_count = (df_clean['cleaned_text'].str.strip() == "").sum()
print(f"Empty texts after cleaning: {empty_count}")

# Original vs Cleaned Text ka side-by-side comparison dekhein
df_clean[[text_col, 'cleaned_text']].head(5)

Empty texts after cleaning: 1


,Consumer Complaint,cleaned_text
0,I closed my {$500.00} limit credit card accoun...,i closed my limit credit card account with in ...
1,I have not been living at the address which ju...,i have not been living at the address which ju...
2,On XXXX XXXX I went to the website of my s...,on i went to the website of my student loan co...
3,Although I have never been late on a payment w...,although i have never been late on a payment w...
4,I home is under contract to be sold and the mo...,i home is under contract to be sold and the mo...


## 2.1 Strategic Category Consolidation (Resolving CFPB Nomenclature Overlaps)
CFPB historically renamed several product categories around 2017:
- `Credit card or prepaid card` $\to$ `Credit card`
- `Checking or savings account` $\to$ `Bank account or service`
- `Credit reporting, credit repair services...` $\to$ `Credit reporting`

Merging these identical product aliases into **7 distinct, non-overlapping financial classes** prevents artificial classification ambiguity and significantly improves model reliability.

In [ ]:
# 3 overlapping pairs ko unke single parent category me merge karte hain
category_mapping_cfpb = {
    'Credit card or prepaid card': 'Credit card',
    'Checking or savings account': 'Bank account or service',
    'Credit reporting, credit repair services, or other personal consumer reports': 'Credit reporting'
}

# Replace in 'Product' column 
df_clean['Product_Cleaned'] = df_clean[target_col].replace(category_mapping_cfpb)

print(f"Original Categories Count     : {df_clean[target_col].nunique()}")
print(f"Consolidated Categories Count : {df_clean['Product_Cleaned'].nunique()}\n")

print("--- 7 Clean Financial Product Categories ---")
print(df_clean['Product_Cleaned'].value_counts())

Original Categories Count     : 10
Consolidated Categories Count : 7

--- 7 Clean Financial Product Categories ---
Product_Cleaned
Credit reporting           55566
Debt collection            46748
Mortgage                   32793
Credit card                22062
Bank account or service    15938
Student loan               12442
Consumer Loan               7087
Name: count, dtype: int64


## 3. Target Variable Label Encoding
Machine learning algorithms require numerical labels. 
Here, we map the 10 categorical financial products into numeric integers ($0$ to $9$) and save the mapping dictionary for model inference and evaluation.

In [ ]:
label_encoder = LabelEncoder()

# Encoding 'Product_Cleaned' (7 classes) 
df_clean['label'] = label_encoder.fit_transform(df_clean['Product_Cleaned'])

label_mapping = {index: label for index, label in enumerate(label_encoder.classes_)}

print("--- 7 Consolidated Categories Mapping ---")
for num, category in label_mapping.items():
    count = (df_clean['label'] == num).sum()
    print(f"Label {num}: {category:<30} ({count:,} samples)")

--- 7 Consolidated Categories Mapping ---
Label 0: Bank account or service        (15,938 samples)
Label 1: Consumer Loan                  (7,087 samples)
Label 2: Credit card                    (22,062 samples)
Label 3: Credit reporting               (55,566 samples)
Label 4: Debt collection                (46,748 samples)
Label 5: Mortgage                       (32,793 samples)
Label 6: Student loan                   (12,442 samples)


## 4. Stratified Train-Validation-Test Splitting
We partition the dataset into:
- **Training Set (80%):** For model learning.
- **Validation Set (10%):** For hyperparameter tuning and model checkpointing.
- **Test Set (10%):** Unseen final benchmark.

We use `stratify=y` to preserve category proportions across all splits.

In [8]:
features = df_clean['cleaned_text']
labels = df_clean['label']

# 1. Pehla Split: 80% Train, 20% Temp (Val + Test)
X_train, X_temp, y_train, y_temp = train_test_split(
    features, 
    labels, 
    test_size=0.20, 
    random_state=42, 
    stratify=labels
)

# 2. Dusra Split: 20% Temp ko aada-aada (10% Val, 10% Test) karte hain
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, 
    y_temp, 
    test_size=0.50, 
    random_state=42, 
    stratify=y_temp
)

print(f"Total Filtered Data : {len(features):,}")
print(f"Training Set (80%)  : {len(X_train):,} samples")
print(f"Validation Set (10%): {len(X_val):,} samples")
print(f"Test Set (10%)      : {len(X_test):,} samples")

Total Filtered Data : 192,636
Training Set (80%)  : 154,108 samples
Validation Set (10%): 19,264 samples
Test Set (10%)      : 19,264 samples


In [9]:
# Proportions (Percentage %)
train_dist = (y_train.value_counts(normalize=True) * 100).round(2)
val_dist = (y_val.value_counts(normalize=True) * 100).round(2)
test_dist = (y_test.value_counts(normalize=True) * 100).round(2)

split_check_df = pd.DataFrame({
    'Train (%)': train_dist,
    'Val (%)': val_dist,
    'Test (%)': test_dist
})

print("--- Class Proportion Check Across Splits ---")
split_check_df

--- Class Proportion Check Across Splits ---


,Train (%),Val (%),Test (%)
label,,,
3,28.85,28.84,28.85
4,24.27,24.27,24.27
5,17.02,17.02,17.03
2,11.45,11.46,11.45
0,8.27,8.27,8.27
6,6.46,6.46,6.46
1,3.68,3.68,3.68


## 5. Export Processed Datasets & Label Mapping
Saving the partitioned datasets into compressed Parquet format and storing the label mapping dictionary as JSON.
This allows modular, instant loading for the upcoming modeling notebook (`03_modeling.ipynb`).

In [10]:
import os
import json

# Processed data save karne ke liye folder banate hain
output_dir = os.path.join('data', 'processed')
os.makedirs(output_dir, exist_ok=True)

# 1. Train, Val, Test DataFrames banate hain
train_df = pd.DataFrame({'text': X_train, 'label': y_train})
val_df   = pd.DataFrame({'text': X_val,   'label': y_val})
test_df  = pd.DataFrame({'text': X_test,  'label': y_test})

# 2. Fast compressed Parquet files me save karte hain
train_path = os.path.join(output_dir, 'train.parquet')
val_path   = os.path.join(output_dir, 'val.parquet')
test_path  = os.path.join(output_dir, 'test.parquet')

train_df.to_parquet(train_path, index=False)
val_df.to_parquet(val_path, index=False)
test_df.to_parquet(test_path, index=False)

# 3. Label mapping dictionary ko JSON me save karte hain (Inference ke liye)
mapping_path = os.path.join(output_dir, 'label_mapping.json')
with open(mapping_path, 'w') as f:
    json.dump(label_mapping, f, indent=4)

print("--- Processed Artifacts Successfully Saved! ---")
print(f"Train Dataset : {train_path} ({len(train_df):,} rows)")
print(f"Val Dataset   : {val_path} ({len(val_df):,} rows)")
print(f"Test Dataset  : {test_path} ({len(test_df):,} rows)")
print(f"Label Mapping : {mapping_path}")

--- Processed Artifacts Successfully Saved! ---
Train Dataset : data\processed\train.parquet (154,108 rows)
Val Dataset   : data\processed\val.parquet (19,264 rows)
Test Dataset  : data\processed\test.parquet (19,264 rows)
Label Mapping : data\processed\label_mapping.json


---
## 🎯 Preprocessing Executive Summary & Deliverables:

1. **Cohort Selection:** Filtered data to Top 10 Product categories and removed duplicate complaints.
2. **Text Sanitization:** Eliminated URLs, CFPB privacy tokens (`XXXX`), non-alphabetic noise, and standardized text casing.
3. **Encoding:** Successfully mapped 10 categorical targets into integer labels ($0$ to $9$) and exported `label_mapping.json`.
4. **Data Splitting:** Applied 80/10/10 Stratified Partitioning, ensuring zero target distribution drift across Train, Validation, and Test sets.
5. **Artifacts Exported:** Serialized clean splits to `data/processed/*.parquet` for fast modeling access.

✅ **Step 2 (Preprocessing) Completed Successfully! Next Phase: Baseline & Advanced NLP Modeling (`03_modeling.ipynb`).**